# YOLOv26 training

This notebook trains a **YOLOv26** (Ultralytics) model on the **Urban Disaster Monitor** dataset for detecting civilians, rescuers and animals in disaster scenarios.

**Google Colab** with a GPU enabled (T4 or better) is recommended.

**Classes:** see `dataset/data.yaml`.

---
**Project:** [Urban Disaster Monitor](https://github.com/MariaCarolinass/urban-disaster-monitor) · **Authors:** Carolina Soares, João Galdino

## Dependencies and GPU

Use the notebook kernel environment and check whether a GPU is available.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "dataset" / "data.yaml").is_file() and (p / "models").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the project repository.")
import torch
print("Dependencies: install app/requirements.txt and notebooks/requirements.txt in the kernel environment.")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Dataset

In [ ]:
DATA_YAML = PROJECT_ROOT / "dataset" / "data.yaml"
if not DATA_YAML.is_file():
    raise FileNotFoundError(DATA_YAML)

## Training

Choose the YOLOv26 variant (`n`, `s`, `m`, `l`) and hyperparameters. The pretrained checkpoint is cached under `notebooks/generated/weights/`; training saves weights and metrics under `runs/detect/<name>`.

In [ ]:
from ultralytics import YOLO
from ultralytics.utils.downloads import safe_download

# Select the starting checkpoint and training parameters for this experiment.
MODEL_VARIANT = "yolo26n.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = 16
RUN_NAME = f"{Path(MODEL_VARIANT).stem}-urban-disaster"
RUNS_DIR = PROJECT_ROOT / "runs" / "detect"

PRETRAINED_MODEL = PROJECT_ROOT / "notebooks" / "generated" / "weights" / MODEL_VARIANT
PRETRAINED_MODEL.parent.mkdir(parents=True, exist_ok=True)
if not PRETRAINED_MODEL.is_file():
    safe_download(
        url=f"https://github.com/ultralytics/assets/releases/latest/download/{MODEL_VARIANT}",
        file=PRETRAINED_MODEL,
        min_bytes=100_000,
    )

model = YOLO(str(PRETRAINED_MODEL))
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(RUNS_DIR),
    name=RUN_NAME,
)

## Validate on images

After training, test the model on validation images.

In [ ]:
weights_path = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
if weights_path.is_file():
    model = YOLO(str(weights_path))
    results = model.predict(source=str(PROJECT_ROOT / "dataset" / "valid" / "images"), save=True)
    print(f"Loaded: {weights_path}")
else:
    print("Run the training cell first." )